# 1. Counting and grouping
Recompute structural Pauli counts and measurement-setting counts for small model spaces. Table I counts symmetry-allowed strings including the identity. Table II begins with coefficients retained for a specific interaction and omits the identity. A stable descending coefficient-magnitude order fixes the greedy tie convention.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
if not (DATA / "structural_counts.tsv").is_file():
    raise FileNotFoundError("Start the notebook with its working directory at the repository root.")
sys.path.insert(0, str(ROOT / "lib"))

def read_data(filename):
    return json.loads((DATA / filename).read_text())

## Table I · structural NN counting
Enumerate allowed two-particle transitions by parity, magnetic projection and common coupled angular momentum. For identical nucleons in the same orbit only even total angular momentum contributes. Deduplicate the resulting supports before counting the allowed X/Y patterns. The diagonal part includes the identity. The Python implementation below is intended for small NN spaces.

In [ ]:
from collections import defaultdict
from functools import lru_cache
from itertools import combinations, product
from math import comb
from sympy import Rational
from sympy.physics.wigner import clebsch_gordan

@lru_cache(maxsize=None)
def allowed_j(a, b, identical_orbit):
    # Angular momenta and their projections are stored as twice their values.
    j, m, k, n = [Rational(v, 2) for v in (a[2], a[3], b[2], b[3])]
    return {J for J in range(int(abs(j - k)), int(j + k) + 1)
            if (not identical_orbit or J % 2 == 0)
            and clebsch_gordan(j, k, J, m, n, m + n) != 0}

def structural_nn(emin, emax):
    orbitals = [(e, l, j, m) for e in range(emin, emax + 1)
                for l in range(e, -1, -2)
                for j in sorted({abs(2*l - 1), 2*l + 1})
                for m in range(-j, j + 1, 2)]
    n = len(orbitals)
    nq = 2*n
    diagonal = 1 + nq + comb(nq, 2)
    # Each allowed radial transition has XX and YY patterns for each species.
    one_body = 4 * sum(a[1:] == b[1:] for a, b in combinations(orbitals, 2))
    def pair_patterns(like_species):
        pairs = combinations(range(n), 2) if like_species else product(range(n), repeat=2)
        channels = defaultdict(list)
        for i, j in pairs:
            a, b = orbitals[i], orbitals[j]
            angular = allowed_j(a, b, like_species and a[:3] == b[:3])
            if angular:
                channels[(a[3] + b[3], (a[1] + b[1]) % 2)].append((i, j, angular))
        supports = {}
        for states in channels.values():
            for (i, j, bra), (k, l, ket) in combinations(states, 2):
                if not bra.intersection(ket):
                    continue
                if like_species:
                    key = tuple(sorted((i, j, k, l)))
                    nxy = 4 if len(set(key)) == 4 else 2
                else:
                    key = (tuple(sorted((i, k))), tuple(sorted((j, l))))
                    nxy = 2 * (int(i != k) + int(j != l))
                supports[key] = 2 ** (nxy - 1)
        return sum(supports.values())
    return nq, diagonal, diagonal + one_body + 2*pair_patterns(True) + pair_patterns(False)

In [ ]:
counts = pd.read_csv(DATA / "structural_counts.tsv", sep="\t")
test_spaces = [("NN", "emax=0", 0, 0), ("valence NN", "p", 1, 1), ("NN", "emax=1", 0, 1)]
structural_results = []
for family, space, emin, emax in test_spaces:
    nq, diagonal, total = structural_nn(emin, emax)
    reference = counts[(counts["Interaction-type"] == family) & (counts.modelspace == space)].iloc[0]
    assert (nq, diagonal, total) == tuple(reference[["Nq", "IZterms", "Num of Hterms"]])
    structural_results.append((family, space, nq, diagonal, total))
pd.DataFrame(structural_results, columns=["interaction", "space", "qubits", "I/Z strings", "structural strings"])

## Table II · coefficient-dependent measurement settings
Each input row represents an ordered even Majorana support and its real coefficient (MeV). Jordan–Wigner conversion gives Pauli X and Z masks. First fit inserts a term into the earliest compatible setting. QWC requires a common local Pauli axis, FC requires mutual commutation, and Majorana pairing requires a common perfect matching. The same-signed or opposite-signed terms are ordered by absolute weight.

The default runs NN at $e_{\max}=0,1$ and the CK interaction in the $p$ shell. Add `3nf_e1_compact_common` to run the bundled 51,376-term NN+3N case. Larger spaces use the results in `grouping.json`.

In [ ]:
from numba import njit


@njit
def operator_bits(s, d, q):
    m = len(d); w = (q+63)//64
    a = np.zeros((m, (2*q+63)//64), np.uint64)
    x = np.zeros((m, w), np.uint64); z = np.zeros_like(x)
    for i in range(m):
        for j in range(d[i]):
            k = int(s[i,j]); p = k//2
            a[i,k//64] ^= np.uint64(1) << np.uint64(k%64)
            x[i,p//64] ^= np.uint64(1) << np.uint64(p%64)
            for t in range(p): z[i,t//64] ^= np.uint64(1) << np.uint64(t%64)
            if k%2: z[i,p//64] ^= np.uint64(1) << np.uint64(p%64)
    return a, x, z


@njit(inline='always')
def parity(v):
    v ^= v >> np.uint64(32); v ^= v >> np.uint64(16)
    v ^= v >> np.uint64(8); v ^= v >> np.uint64(4)
    return (np.uint64(0x6996) >> (v & np.uint64(15))) & np.uint64(1)


@njit
def fc_insertion(a, order, q):
    """Coefficient-sorted first fit, against a GF(2) basis, without an M x M graph."""
    w = a.shape[1]; capacity = 32; ng = 0
    basis = np.zeros((capacity, q, w), np.uint64)
    pivots = np.full((capacity, q), -1, np.int32); ranks = np.zeros(capacity, np.int32)
    ids = np.full(len(a), -1, np.int32)
    for i in order:
        found = ng
        for g in range(ng):
            good = True
            for r in range(ranks[g]):
                v = np.uint64(0)
                for b in range(w): v ^= a[i,b] & basis[g,r,b]
                if parity(v): good = False; break
            if good: found = g; break
        if found == ng:
            if ng == capacity:
                new = np.zeros((2*capacity, q, w), np.uint64); new[:capacity] = basis; basis = new
                pp = np.full((2*capacity, q), -1, np.int32); pp[:capacity] = pivots; pivots = pp
                rr = np.zeros(2*capacity, np.int32); rr[:capacity] = ranks; ranks = rr; capacity *= 2
            ng += 1
        g = found; ids[i] = g; v = a[i].copy()
        for r in range(ranks[g]):
            p = pivots[g,r]
            if (v[p//64] >> np.uint64(p%64)) & np.uint64(1):
                for b in range(w): v[b] ^= basis[g,r,b]
        pivot = -1
        for p in range(2*q-1, -1, -1):
            if (v[p//64] >> np.uint64(p%64)) & np.uint64(1): pivot = p; break
        if pivot >= 0:
            r = ranks[g]; assert r < q
            while r > 0 and pivots[g,r-1] < pivot:
                basis[g,r] = basis[g,r-1]; pivots[g,r] = pivots[g,r-1]; r -= 1
            basis[g,r] = v; pivots[g,r] = pivot; ranks[g] += 1
    return ids, basis[:ng], ranks[:ng]


@njit
def qwc_insertion(x, z, order):
    capacity = 32; ng = 0; w = x.shape[1]
    bx = np.zeros((capacity, w), np.uint64); bz = np.zeros_like(bx)
    ids = np.empty(len(x), np.int32)
    for i in order:
        g = 0
        while g < ng:
            good = True
            for b in range(w):
                if ((x[i,b] ^ bx[g,b]) | (z[i,b] ^ bz[g,b])) & (x[i,b] | z[i,b]) & (bx[g,b] | bz[g,b]):
                    good = False; break
            if good: break
            g += 1
        if g == ng:
            if ng == capacity:
                xx = np.zeros((2*capacity,w), np.uint64); xx[:capacity] = bx; bx = xx
                zz = np.zeros_like(xx); zz[:capacity] = bz; bz = zz; capacity *= 2
            ng += 1
        ids[i] = g
        for b in range(w): bx[g,b] |= x[i,b]; bz[g,b] |= z[i,b]
    return ids, bx[:ng], bz[:ng]


@njit(inline='always')
def matching_compatible(row, degree, mate):
    for j in range(degree):
        partner = mate[row[j]]
        if partner < 0: continue
        found = False
        for k in range(degree):
            if row[k] == partner: found = True; break
        if not found: return False
    return True


@njit
def pairing_insertion(s, d, order, q):
    """First-fit partial matchings; pair free labels in sorted order.

    This is a complete support-aware heuristic for degrees 2, 4, AND 6.
    It is not the published universal two-body schedule.
    """
    capacity = 32; ng = 0; mates = np.full((capacity, 2*q), -1, np.int16)
    ids = np.full(len(d), -1, np.int32)
    for i in order:
        g = 0
        while g < ng and not matching_compatible(s[i], d[i], mates[g]): g += 1
        if g == ng:
            if ng == capacity:
                new = np.full((2*capacity,2*q), -1, np.int16); new[:capacity] = mates; mates = new; capacity *= 2
            ng += 1
        ids[i] = g; previous = -1
        for j in range(d[i]):
            v = s[i,j]
            if mates[g,v] >= 0: continue
            if previous < 0: previous = v
            else: mates[g,previous] = v; mates[g,v] = previous; previous = -1
        assert previous == -1
    for g in range(ng):
        previous = -1
        for v in range(2*q):
            if mates[g,v] >= 0: continue
            if previous < 0: previous = v
            else: mates[g,previous] = v; mates[g,v] = previous; previous = -1
    return ids, mates[:ng]


@njit
def verify_matching(s,d,ids,mates):
    for i in range(len(d)):
        for j in range(d[i]):
            partner=mates[ids[i],s[i,j]];count=0
            for k in range(d[i]):
                if s[i,k]==partner:count+=1
            assert count==1


In [ ]:
case_ids = ["nn_e0", "valence_p", "nn_e1"]
published = {r["id"]: r for r in read_data("grouping.json")}
operator_ref = {r["id"]: r for r in read_data("operator_summaries.json")["rows"]}

def group_case(archive, case):
    get = lambda key: archive[f"{case}__{key}"]
    supports, degree, h = get("supports"), get("degree"), get("coeffs_majorana")
    nq = int(get("nqubits"))
    assert np.all(degree == (supports >= 0).sum(axis=1)) and np.all(h != 0)
    masks, x, z = operator_bits(supports, degree, nq)
    order = np.argsort(-np.abs(h), kind="stable")
    qwc, basis_x, basis_z = qwc_insertion(x, z, order)
    fc, generators, ranks = fc_insertion(masks, order, nq)
    # For even Majorana monomials the binary dot product is their commutator parity.
    # Test group generators in blocks so the optional NN+3N case uses bounded memory.
    parity_byte = np.array([n.bit_count() % 2 for n in range(256)], dtype=np.uint8)
    for start in range(0, len(h), 1024):
        stop = min(start + 1024, len(h))
        overlap = masks[start:stop, None, :] & generators[fc[start:stop]]
        parity = np.bitwise_xor.reduce(parity_byte[overlap.view(np.uint8)], axis=-1)
        assert not np.any(parity)
    paired, mates = pairing_insertion(supports, degree, order, nq)
    verify_matching(supports, degree, paired, mates)
    occupied = (x | z) & (basis_x[qwc] | basis_z[qwc])
    assert not np.any(((x ^ basis_x[qwc]) | (z ^ basis_z[qwc])) & occupied)
    for method, ids in {"qwc": qwc, "fc": fc, "pairing_greedy": paired}.items():
        assert np.array_equal(np.unique(ids), np.arange(published[case]["methods"][method]["settings"]))
    assert len(h) == published[case]["nonidentity_terms"]
    return [case, nq, len(h), len(basis_x), len(ranks), len(mates)]

with np.load(DATA / "small_operators.npz", allow_pickle=False) as archive:
    group_results = [group_case(archive, case) for case in case_ids]
pd.DataFrame(group_results, columns=["case", "qubits", "retained terms", "QWC", "FC greedy", "Majorana greedy"])

## Table IV · direct commutator sum
For even Majorana monomials, two terms anticommute when their supports intersect in an odd number of labels. Sum $|h_i h_j|$ once per such pair. This direct check is quadratic, so it runs only for the three small NN examples.

In [ ]:
import math

with np.load(DATA / "small_operators.npz", allow_pickle=False) as archive:
    for case in ("nn_e0", "valence_p", "nn_e1"):
        rows = archive[f"{case}__supports"]
        weights = np.abs(archive[f"{case}__coeffs_majorana"]).tolist()
        masks = [sum(1 << int(k) for k in row if k >= 0) for row in rows]
        coefficient = math.fsum(weights[j]*weights[i] for j, b in enumerate(masks)
                                for i in range(j) if (masks[i] & b).bit_count() % 2)
        np.testing.assert_allclose(coefficient, operator_ref[case]["C"], rtol=1e-11)
        print(f"{case:12} C_pair = {coefficient:.8g} MeV²")

## Optional · original Julia counter, including NN+3N
The source file in `lib/` is the one identified by the SHA-256 in the structural TSV. For the optional cell install Julia 1.12 and instantiate the project as described in the README. Set `run_julia` to `True` to recalculate a small NN+3N row. Set `full_sweep` to `True` as well to reproduce all structural rows. Results go to `build/`.

In [ ]:
import subprocess
run_julia = False
full_sweep = False
if run_julia:
    (ROOT / "build").mkdir(exist_ok=True)
    spaces = "configured_spaces()" if full_sweep else "[CountConfig(emax=1, is_3NF=true, e3max=3)]"
    command = f'include(ARGS[1]); export_counts({spaces}; output_path=ARGS[2])'
    output = ROOT / "build" / "recount.tsv"
    subprocess.run(["julia", "--startup-file=no", "--project=lib", "--threads=2", "-e", command,
                    str(ROOT / "lib/counting_Hamiltonian_terms.jl"), str(output)], check=True)
    keys = ["Interaction-type", "modelspace"]
    columns = ["Nq", "IZterms", "Num of Hterms"]
    actual = pd.read_csv(output, sep="\t").set_index(keys)[columns]
    pd.testing.assert_frame_equal(actual, counts.set_index(keys).loc[actual.index, columns])